In [1]:
import sys
print(sys.executable)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

engine = create_engine("postgresql+psycopg2://localhost/olist")
sns.set_theme(style="whitegrid")

/Users/aminentezari/Desktop/Projects_python/olist-customer-analytics/.venv/bin/python


In [2]:
query = """
WITH main_item AS (
    -- the most expensive item defines the order's main category and seller
    SELECT DISTINCT ON (i.order_id)
           i.order_id, p.category, s.seller_id, s.seller_state
    FROM fact_order_items i
    JOIN dim_product p ON p.product_id = i.product_id
    JOIN dim_seller  s ON s.seller_id  = i.seller_id
    ORDER BY i.order_id, i.price DESC
),
review_time AS (
    SELECT DISTINCT ON (order_id) order_id, review_creation_date
    FROM order_reviews
    ORDER BY order_id, review_creation_date DESC, review_answer_timestamp DESC
)
SELECT f.order_id,
       f.order_date,
       f.delivered_date,
       f.estimated_date,
       rt.review_creation_date::date AS review_date,
       f.review_score,
       f.is_bad_review,
       f.days_vs_estimate,
       f.delivery_days,
       (f.estimated_date - f.order_date) AS promised_days,
       f.revenue,
       f.freight_value,
       f.n_items,
       f.n_sellers,
       f.main_payment_type,
       f.installments,
       c.customer_state,
       m.category,
       m.seller_id,
       m.seller_state
FROM fact_orders f
JOIN dim_customer c ON c.customer_id = f.customer_id
JOIN main_item m    ON m.order_id = f.order_id
JOIN review_time rt ON rt.order_id = f.order_id
WHERE f.review_score IS NOT NULL
  AND f.delivered_date IS NOT NULL
"""
df = pd.read_sql(query, engine)
for col in ["order_date", "delivered_date", "estimated_date", "review_date"]:
    df[col] = pd.to_datetime(df[col])

df["target"] = df["is_bad_review"].astype(int)
df["freight_ratio"] = df["freight_value"] / df["revenue"]

print(df.shape)
print(df["target"].value_counts(normalize=True).round(3))
df.head()

(95560, 22)
target
0    0.872
1    0.128
Name: proportion, dtype: float64


,order_id,order_date,delivered_date,estimated_date,review_date,review_score,is_bad_review,days_vs_estimate,delivery_days,promised_days,...,n_items,n_sellers,main_payment_type,installments,customer_state,category,seller_id,seller_state,target,freight_ratio
0,00010242fe8c5a6d1ba2dd792cb16214,2017-09-13,2017-09-20,2017-09-29,2017-09-21,5,False,-9,7.614421,16,...,1,1,credit_card,2,RJ,cool_stuff,48436dade18ac8b2bce089ec2a041202,SP,0,0.225637
1,00018f77f2f0320c557190d7a144bdd3,2017-04-26,2017-05-12,2017-05-15,2017-05-13,4,False,-3,16.216181,19,...,1,1,credit_card,3,SP,pet_shop,dd7ddc04e1b6c2c614352b383efe2d36,SP,0,0.083076
2,000229ec398224ef6ca0657da4fc703e,2018-01-14,2018-01-22,2018-02-05,2018-01-23,5,False,-14,7.948437,22,...,1,1,credit_card,5,MG,furniture_decor,5b51032eddd242adc84c38acab88f23d,MG,0,0.089799
3,00024acbcdf0a6daa1e931b038114c75,2018-08-08,2018-08-14,2018-08-20,2018-08-15,4,False,-6,6.147269,12,...,1,1,credit_card,2,SP,perfumery,9d7a1d34a5052409006425275ba1c2b4,SP,0,0.984604
4,00048cc3ae777c65dbb7d2a0634bc1ea,2017-05-15,2017-05-22,2017-06-06,2017-05-23,4,False,-15,6.668067,22,...,1,1,boleto,1,MG,housewares,6426d21aca402a131fc0a5d0960a3c90,SP,0,0.579452


In [7]:
early = df["review_date"] < df["delivered_date"]
print("Reviews created before delivery:", early.sum(), f"({100*early.mean():.1f}%)")
print("Bad-review rate, review before delivery:", round(df.loc[early, "target"].mean(), 3))
print("Bad-review rate, review after delivery: ", round(df.loc[~early, "target"].mean(), 3))

Reviews created before delivery: 0 (0.0%)
Bad-review rate, review before delivery: nan
Bad-review rate, review after delivery:  0.095


In [3]:
split_date = pd.Timestamp("2018-04-01")
train = df[df["order_date"] <  split_date].copy()
test  = df[df["order_date"] >= split_date].copy()

summary = pd.DataFrame({
    "orders":   [len(train), len(test)],
    "bad_rate": [train["target"].mean(), test["target"].mean()],
    "late_rate": [(train["days_vs_estimate"] > 0).mean(), (test["days_vs_estimate"] > 0).mean()],
    "from": [train["order_date"].min().date(), test["order_date"].min().date()],
    "to":   [train["order_date"].max().date(), test["order_date"].max().date()],
}, index=["train", "test"]).round(3)
summary

,orders,bad_rate,late_rate,from,to
train,63566,0.140,0.078,2017-01-05,2018-03-31
test,31994,0.104,0.043,2018-04-01,2018-08-29


In [4]:
early = df["review_date"] < df["delivered_date"]
print("Reviews before delivery:", early.sum())

pd.crosstab(early, df["days_vs_estimate"] > 0,
            rownames=["review_before_delivery"], colnames=["is_late"],
            normalize="index").round(3)

Reviews before delivery: 4960


is_late,False,True
review_before_delivery,,
False,0.982,0.018
True,0.043,0.957


In [8]:
df = df[~early].copy()
print(df.shape)
print(df["target"].value_counts(normalize=True).round(3))

(90600, 22)
target
0    0.905
1    0.095
Name: proportion, dtype: float64


/var/folders/4k/1xlcs6f53xx7nh0n22bbbxr00000gq/T/ipykernel_58746/779760394.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~early].copy()


In [9]:
split_date = pd.Timestamp("2018-04-01")
train = df[df["order_date"] <  split_date].copy()
test  = df[df["order_date"] >= split_date].copy()

pd.DataFrame({
    "orders":    [len(train), len(test)],
    "bad_rate":  [train["target"].mean(), test["target"].mean()],
    "late_rate": [(train["days_vs_estimate"] > 0).mean(), (test["days_vs_estimate"] > 0).mean()],
}, index=["train", "test"]).round(3)


,orders,bad_rate,late_rate
train,59536,0.099,0.019
test,31064,0.087,0.015


In [10]:
hist = pd.read_sql("""
WITH rt AS (
    SELECT DISTINCT ON (order_id) order_id, review_creation_date::date AS review_date
    FROM order_reviews
    ORDER BY order_id, review_creation_date DESC, review_answer_timestamp DESC
)
SELECT DISTINCT i.order_id, i.seller_id, rt.review_date,
       f.is_bad_review::int AS bad,
       COALESCE((f.days_vs_estimate > 0)::int, 0) AS late
FROM fact_order_items i
JOIN fact_orders f ON f.order_id = i.order_id
JOIN rt ON rt.order_id = f.order_id
WHERE f.review_score IS NOT NULL
""", engine)
hist["review_date"] = pd.to_datetime(hist["review_date"])

daily = (hist.groupby(["seller_id", "review_date"])
             .agg(n=("bad", "size"), bad=("bad", "sum"), late=("late", "sum"))
             .reset_index()
             .sort_values("review_date"))
daily[["cum_n", "cum_bad", "cum_late"]] = daily.groupby("seller_id")[["n", "bad", "late"]].cumsum()
daily = daily.rename(columns={"review_date": "hist_date"})

df = df.sort_values("order_date")
df = pd.merge_asof(df, daily[["seller_id", "hist_date", "cum_n", "cum_bad", "cum_late"]],
                   left_on="order_date", right_on="hist_date", by="seller_id",
                   allow_exact_matches=False)
df[["cum_n", "cum_bad", "cum_late"]] = df[["cum_n", "cum_bad", "cum_late"]].fillna(0)

prior = df.loc[df["order_date"] < split_date, "target"].mean()
m = 10
df["seller_past_reviews"]  = df["cum_n"]
df["seller_past_bad_rate"] = (df["cum_bad"] + prior * m) / (df["cum_n"] + m)
df["seller_past_late_rate"] = df["cum_late"] / df["cum_n"].replace(0, np.nan)
df["seller_past_late_rate"] = df["seller_past_late_rate"].fillna(0)

print(df.shape)
df[["seller_past_reviews", "seller_past_bad_rate", "seller_past_late_rate"]].describe().round(3)

(90600, 29)


,seller_past_reviews,seller_past_bad_rate,seller_past_late_rate
count,90600.000,90600.000,90600.000
mean,168.091,0.116,0.045
std,283.265,0.054,0.067
min,0.000,0.013,0.000
25%,11.000,0.082,0.000
50%,50.000,0.103,0.031
75%,176.000,0.143,0.061
max,1775.000,0.564,1.000


In [12]:
train = df[df["order_date"] <  split_date].copy()
test  = df[df["order_date"] >= split_date].copy()

num_features = ["days_vs_estimate", "delivery_days", "promised_days",
                "revenue", "freight_value", "freight_ratio",
                "n_items", "n_sellers", "installments",
                "seller_past_reviews", "seller_past_bad_rate", "seller_past_late_rate"]
cat_features = ["main_payment_type", "customer_state", "category", "seller_state"]

X_train, y_train = train[num_features + cat_features], train["target"]
X_test,  y_test  = test[num_features + cat_features],  test["target"]

print(X_train.shape, X_test.shape)
print(X_train.isna().sum().sum(), X_test.isna().sum().sum())

(59536, 16) (31064, 16)
0 0


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), cat_features),
])

logreg = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(class_weight="balanced", max_iter=2000)),
])
logreg.fit(X_train, y_train)

def evaluate(model, name):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(name)
    print("  precision:", round(precision_score(y_test, pred), 3))
    print("  recall:   ", round(recall_score(y_test, pred), 3))
    print("  F1:       ", round(f1_score(y_test, pred), 3))
    print("  ROC-AUC:  ", round(roc_auc_score(y_test, proba), 3))
    print("  PR-AUC:   ", round(average_precision_score(y_test, proba), 3))
    print("  confusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(y_test, pred))
    return proba

proba_logreg = evaluate(logreg, "Logistic regression")

Logistic regression
  precision: 0.167
  recall:    0.492
  F1:        0.25
  ROC-AUC:   0.675
  PR-AUC:    0.202
  confusion matrix [[TN, FP], [FN, TP]]:
[[21703  6646]
 [ 1378  1337]]


In [14]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder

preprocess_tree = ColumnTransformer([
    ("num", "passthrough", num_features),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                           encoded_missing_value=-1), cat_features),
])

gbm = Pipeline([
    ("prep", preprocess_tree),
    ("model", HistGradientBoostingClassifier(
        class_weight="balanced",
        categorical_features=list(range(len(num_features), len(num_features) + len(cat_features))),
        learning_rate=0.05, max_iter=300, max_leaf_nodes=31,
        random_state=42)),
])
gbm.fit(X_train, y_train)

proba_gbm = evaluate(gbm, "Gradient boosting")

Gradient boosting
  precision: 0.177
  recall:    0.436
  F1:        0.252
  ROC-AUC:   0.668
  PR-AUC:    0.208
  confusion matrix [[TN, FP], [FN, TP]]:
[[22858  5491]
 [ 1532  1183]]


In [15]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(logreg, X_test, y_test,
                              scoring="average_precision",
                              n_repeats=5, random_state=42, n_jobs=-1)

importance = (pd.DataFrame({"feature": X_test.columns,
                            "pr_auc_drop": perm.importances_mean,
                            "std": perm.importances_std})
                .sort_values("pr_auc_drop", ascending=False)
                .round(4))
importance

,feature,pr_auc_drop,std
6,n_items,0.0478,0.0024
7,n_sellers,0.0357,0.0018
2,promised_days,0.0243,0.0005
10,seller_past_bad_rate,0.0195,0.0016
0,days_vs_estimate,0.0125,0.0008
13,customer_state,0.0043,0.0004
14,category,0.0029,0.0013
1,delivery_days,0.0012,0.0002
8,installments,0.0009,0.0001
11,seller_past_late_rate,0.0009,0.0004


In [16]:
for col in ["n_items", "n_sellers"]:
    t = (train.assign(group=train[col].clip(upper=3))
              .groupby("group")["target"]
              .agg(orders="size", bad_rate="mean")
              .round(3))
    print(col, "(3 = 3 or more)")
    print(t, "\n")

n_items (3 = 3 or more)
       orders  bad_rate
group                  
1       53514     0.082
2        4645     0.237
3        1377     0.314 

n_sellers (3 = 3 or more)
       orders  bad_rate
group                  
1       58837     0.095
2         666     0.474
3          33     0.697 



In [17]:
rank = pd.DataFrame({"proba": proba_logreg, "bad": y_test.values}).sort_values("proba", ascending=False)
total_bad = rank["bad"].sum()

rows = []
for share in [0.05, 0.10, 0.20]:
    top = rank.head(int(len(rank) * share))
    rows.append({"contact_top": f"{int(share*100)}%",
                 "orders_contacted": len(top),
                 "bad_reviews_caught": int(top["bad"].sum()),
                 "precision": round(top["bad"].mean(), 3),
                 "recall": round(top["bad"].sum() / total_bad, 3),
                 "lift": round(top["bad"].mean() / rank["bad"].mean(), 2)})
pd.DataFrame(rows)

,contact_top,orders_contacted,bad_reviews_caught,precision,recall,lift
0,5%,1553,500,0.322,0.184,3.68
1,10%,3106,778,0.250,0.287,2.87
2,20%,6212,1155,0.186,0.425,2.13


## Findings

**Goal.** Predict whether a delivered order will get a bad review (1 or 2 stars), using only information available at delivery time, so customer care can contact at-risk customers before they write the review.

**Data decisions**
- 95,560 delivered orders with a review, January 2017 to August 2018.
- 4,960 reviews (5.2%) were written before the parcel arrived, and 95.7% of those orders were late. At delivery time these reviews already exist, so they were removed from the model. For these orders a simple rule is enough: alert customer care when an order passes its estimated date undelivered.
- Final dataset: 90,600 orders, 9.5% bad reviews.
- Time-based split: train on orders before 2018-04-01 (59,536), test on April to August 2018 (31,064).
- Seller history features use only reviews written before each order was placed, to avoid leakage.

**Models**

| Metric | Logistic regression | Gradient boosting |
|---|---|---|
| ROC-AUC | 0.675 | 0.668 |
| PR-AUC | 0.202 | 0.208 |

The random baseline for PR-AUC is 0.087. Gradient boosting did not beat logistic regression, so I kept the simpler model. The limit is the information in the features, not the algorithm: product quality and wrong items are not in this data.

**What drives bad reviews**
- Number of items and number of sellers are the strongest features.
- Orders with 2 items get bad reviews 23.7% of the time, against 8.2% for 1 item.
- Orders with 2 sellers get bad reviews 47.4% of the time, against 9.5% for 1 seller. This is consistent with split shipments, but the data cannot prove the cause.
- Promised delivery time and the seller's past bad-review rate also matter.

**How the business would use it**

Rank orders by risk and contact the riskiest ones:

| Contact the top | Orders | Precision | Recall | Lift |
|---|---|---|---|---|
| 5% | 1,553 | 32.2% | 18.4% | 3.68x |
| 10% | 3,106 | 25.0% | 28.7% | 2.87x |
| 20% | 6,212 | 18.6% | 42.5% | 2.13x |

Contacting the top 5% means about 10 orders a day, and 1 in 3 of them was heading for a bad review.